# Scikit SGD: a small milk-yield model

We will predict **daily milk yield** from days in milk, parity, and previous yield. The JSON records are tiny synthetic examples, not a production dataset or evidence of farm-level accuracy.

The native model uses `SGDRegressor.partial_fit()`. Everything around it uses the same Bovi Core contracts as the other tutorials.

**Route:** inspect data → configure → train → inspect learning → evaluate → save → start another attempt.

Run the notebook from top to bottom with the repository's Python 3.12 environment after `just sync`. No GPU, cloud account, or downloaded weights are needed.

## 1. Open the experiment

The [experiment YAML](../../../data/experiments/scikit_sgd/versions/v1/config/config.yaml) selects the files and pipeline settings. It is input syntax, not the trainer's runtime API: we convert its model, training, and evaluation sections into separate typed objects.

The output directory below is only for this tutorial. Set `BOVI_NOTEBOOK_OUTPUT_DIR` to keep results somewhere durable; the default uses the system temporary directory.

In [ ]:
import os
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import polars as pl
from bovi_core.config import Config
from IPython.display import display

Config.reset()
config = Config(experiment_name="scikit_sgd", project_name="scikit-sgd")
output_root = Path(os.environ.get("BOVI_NOTEBOOK_OUTPUT_DIR", gettempdir()))
output_dir = output_root / "bovi-scikit-sgd" / str(uuid4())

In [ ]:
from scikit_sgd import (
    ScikitSGDEvaluationConfig,
    ScikitSGDModelConfig,
    ScikitSGDTrainingConfig,
)

model_config = ScikitSGDModelConfig.from_config(config)
training_config = ScikitSGDTrainingConfig.from_config(config)
evaluation_config = ScikitSGDEvaluationConfig.from_config(config)

display(
    pl.DataFrame(
        [
            {"config": group, "setting": name, "value": str(value)}
            for group, settings in (
                ("model", model_config),
                ("training", training_config),
                ("evaluation", evaluation_config),
            )
            for name, value in settings.model_dump().items()
        ]
    )
)

**Three different questions:** the model config says *what model to construct*; the training config says *how to update it in this attempt*; the evaluation config says *which metrics to calculate*.

You can also construct `ScikitSGDTrainingConfig(epochs=10, learning_rate=0.01)` directly. We use YAML here so the complete experiment stays visible in one place. This does not yet make the legacy dataloader constructor independent of Bovi Config.

## 2. Follow one record into a batch

The factory connects **source → transforms → dataset → loader**. The source reads JSON records; the dataset selects features and labels; the loader groups them into batches.

This experiment clips input ranges and scales the three features. Labels remain in kilograms. The table below shows raw values first, then the actual model inputs.

In [ ]:
from scikit_sgd import create_dataloader

dataloaders = {
    split: create_dataloader(config, model_config, split) for split in ("train", "validation")
}
train_loader = dataloaders["train"]
validation_loader = dataloaders["validation"]
source = train_loader.dataset.source

display(
    pl.DataFrame(
        [
            {
                "split": split,
                "records": loader.num_samples,
                "batches": len(loader),
                "batch_size": loader.batch_size,
                "loader": type(loader).__name__,
            }
            for split, loader in dataloaders.items()
        ]
    )
)
display(pl.DataFrame([source.source.load_item(index) for index in range(min(4, len(source)))]))

In [ ]:
batch = next(iter(train_loader))

display(
    pl.DataFrame(
        [
            {"field": f"features.{name}", "shape": str(values.shape), "dtype": str(values.dtype)}
            for name, values in batch["features"].items()
        ]
        + [
            {
                "field": "labels",
                "shape": str(batch["labels"].shape),
                "dtype": str(batch["labels"].dtype),
            }
        ]
    )
)

display(
    pl.DataFrame(
        {
            **{name: np.asarray(values).reshape(-1) for name, values in batch["features"].items()},
            "target": np.asarray(batch["labels"]).reshape(-1),
        }
    )
)

The named feature mapping is the shared batch contract. Scikit receives NumPy arrays. The model's configured feature order controls the final matrix, not dictionary order. Metadata travels beside the inputs and is not an extra model feature.

## 3. Construct a fresh model and one attempt

The provider creates a model with no checkpoint. The trainer receives that model, the existing loaders, and the immutable training config.

A `TrainingContext` records the attempt identity and output location. It does not contain learning-rate or optimizer settings.

In [ ]:
from bovi_core.ml import TrainingContext
from scikit_sgd import ScikitSGDModelProvider, ScikitSGDTrainer

provider = ScikitSGDModelProvider()
model = provider.create(model_config)
context = TrainingContext(
    run_id=uuid4(),
    reason="CPU tutorial: first attempt",
    output_dir=output_dir / "first",
)
trainer = ScikitSGDTrainer(
    model=model,
    dataloaders=dataloaders,
    config=training_config,
    context=context,
)
print(f"{type(model).__name__} wraps {type(model.native_model).__name__}")

## 4. Train, then read the learning curve

`train()` returns a lightweight result, not another copy of the model. The updated model remains available as `trainer.model`. Each recorded epoch contains metrics measured after its updates.

In [ ]:
training_result = trainer.train()
assert training_result.status == "completed", training_result.issues
assert training_result.epochs, "Expected completed epochs"

display(
    pl.DataFrame(
        [
            {
                "status": str(training_result.status),
                "stop_reason": str(training_result.stop_reason),
                "epochs": len(training_result.epochs),
                "best_epoch": training_result.best_epoch,
                "seconds": (
                    training_result.completed_at - training_result.started_at
                ).total_seconds(),
                "training_records": training_result.num_examples,
                "sample_exposures": training_result.num_examples_processed,
            }
        ]
    )
)

In [ ]:
import matplotlib.pyplot as plt

history = pl.DataFrame(
    [{"epoch": epoch.epoch, **epoch.metrics} for epoch in training_result.epochs]
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["epoch"], history["train_mse"], label="Training MSE")
ax.plot(history["epoch"], history["validation_mse"], label="Validation MSE")
if training_result.best_epoch is not None:
    ax.axvline(training_result.best_epoch, color="black", linestyle=":", label="Best epoch")
ax.set(xlabel="Epoch in this attempt", ylabel="Mean squared error", title="Did learning improve?")
ax.legend()
fig.tight_layout()
plt.show()

display(history.head(3))
display(history.tail(3))
assert history["train_mse"][-1] < history["train_mse"][0]

**How to read this:** lower MSE is better. Validation uses separate records, but it is also used to select the best epoch, so this is not a final held-out test score.

The final epoch and best epoch can differ. Small improvements still count for the **best checkpoint**; `min_delta` only governs early-stopping patience. Sample exposures count repeated updates across epochs, not unique animals.

## 5. Evaluate the last model independently

The evaluator receives a model and a validation loader, not a `TrainingResult`. We evaluate the **last in-memory model** here. Evaluating the best model would first require restoring `best_checkpoint` into a separate model.

In [ ]:
from bovi_core.ml import EvaluationContext
from scikit_sgd import ScikitSGDEvaluator

evaluation_context = EvaluationContext(
    evaluation_id=uuid4(),
    split="validation",
    model_version="tutorial-last",
    training_run_id=context.run_id,
    output_dir=output_dir / "evaluation",
)
evaluator = ScikitSGDEvaluator(model=model, config=evaluation_config)
evaluation_result = evaluator.evaluate(validation_loader, evaluation_context)
assert evaluation_result.status == "completed", evaluation_result.issues

display(
    pl.DataFrame(
        [{"metric": name, "value": value} for name, value in evaluation_result.metrics.items()]
    )
)

In [ ]:
from bovi_core.ml.dataloaders.model_inputs.numpy_regression import prepare_numpy_regression_inputs

example = validation_loader.dataset.get_input_example(
    n_samples=validation_loader.num_samples,
)
x_validation, y_validation = prepare_numpy_regression_inputs(example, model_config.feature_names)
predictions = np.asarray(model(x_validation)).reshape(-1)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(y_validation, predictions, label="Validation records")
limits = [min(y_validation.min(), predictions.min()), max(y_validation.max(), predictions.max())]
ax.plot(limits, limits, linestyle="--", color="black", label="Perfect prediction")
ax.set(xlabel="Observed target", ylabel="Predicted target", title="Prediction versus target")
ax.legend()
fig.tight_layout()
plt.show()

## 6. Save the result separately from training

Checkpoint files were written during training. The logger now persists the **context, result, and selected config snapshots** as a JSON manifest. It does not export a deployment model.

Logging has its own status: a write failure must not turn a completed training result into a failed training result. The assertion below verifies this tutorial's local write succeeded; it is not a trainer policy.

In [ ]:
from bovi_core.ml.trainers import LocalTrainingResultLogger

run_metadata = {
    "experiment": "scikit_sgd",
    "splits": {split: {"records": loader.num_samples} for split, loader in dataloaders.items()},
}
config_snapshot = {
    "model": model_config.model_dump(mode="json"),
    "training": training_config.model_dump(mode="json"),
    "evaluation": evaluation_config.model_dump(mode="json"),
}
logger = LocalTrainingResultLogger(metadata=run_metadata, config_snapshot=config_snapshot)
log_outcome = await logger.log(context, training_result)

display(
    pl.DataFrame(
        [
            {"destination": item.destination, "status": str(item.status), "location": item.location}
            for item in log_outcome.destinations
        ]
    )
)
assert log_outcome.destinations[0].status == "success", log_outcome

## 7. Reload the saved result and start another attempt

A reference identifies a checkpoint bundle; the resolver verifies it and finds its native payload. The provider then restores a **new model**.

These examples support starting from saved weights, not exact restoration of all optimizer, RNG, or early-stopping state. A new attempt gets a new run ID and starts counting epochs at one.

In [ ]:
import json

from bovi_core.ml.models.checkpoints import LocalCheckpointResolver
from bovi_core.ml.trainers import TrainingResult

manifest_path = context.output_dir / "training-results" / f"{context.run_id}.json"
saved_manifest = json.loads(manifest_path.read_text())
saved_result = TrainingResult.model_validate(saved_manifest["result"])
saved_context = TrainingContext.model_validate(saved_manifest["context"])
saved_model_config = ScikitSGDModelConfig.model_validate(saved_manifest["config_snapshot"]["model"])
assert saved_result == training_result
assert saved_context == context
assert saved_result.last_checkpoint is not None

resolved_checkpoint = LocalCheckpointResolver().resolve(saved_result.last_checkpoint)
resumed_model = provider.restore_checkpoint(saved_model_config, resolved_checkpoint)

In [ ]:
resume_config = ScikitSGDTrainingConfig.model_validate(
    {
        **saved_manifest["config_snapshot"]["training"],
        "epochs": 2,
        "early_stopping_patience": None,
    }
)
resume_context = TrainingContext(
    run_id=uuid4(),
    resumed_from_run_id=saved_context.run_id,
    reason="CPU tutorial: new attempt from saved weights",
    output_dir=output_dir / "resume",
)
resumed_trainer = ScikitSGDTrainer(
    model=resumed_model,
    dataloaders=dataloaders,
    config=resume_config,
    context=resume_context,
)
resume_result = resumed_trainer.train()
assert resume_result.status == "completed", resume_result.issues
assert resume_result.epochs[0].epoch == 1

display(
    pl.DataFrame(
        [
            {
                "attempt": "first",
                "epochs": len(training_result.epochs),
                "final_validation_mse": training_result.epochs[-1].metrics["validation_mse"],
            },
            {
                "attempt": "resumed",
                "epochs": len(resume_result.epochs),
                "final_validation_mse": resume_result.epochs[-1].metrics["validation_mse"],
            },
        ]
    )
)

This second attempt is logged independently. Reusing the original run ID would incorrectly imply that two different outcomes were the same run.

In [ ]:
resume_logger = LocalTrainingResultLogger(
    metadata=run_metadata,
    config_snapshot={**config_snapshot, "training": resume_config.model_dump(mode="json")},
)
resume_log_outcome = await resume_logger.log(resume_context, resume_result)
assert resume_log_outcome.destinations[0].status == "success", resume_log_outcome

resume_manifest_path = (
    resume_context.output_dir / "training-results" / f"{resume_context.run_id}.json"
)
resume_manifest = json.loads(resume_manifest_path.read_text())
assert resume_manifest["context"]["resumed_from_run_id"] == str(context.run_id)
assert TrainingResult.model_validate(resume_manifest["result"]) == resume_result
print(f"Two attempt manifests and their checkpoints are stored under: {output_dir}")

## What this example proves

- The configured records reach a native scikit-learn model through Bovi Core.
- Training returns epoch history and best/last checkpoint references.
- Evaluation and result logging can run independently of the trainer.
- A saved result can be read back and its weights can seed another attempt.

It does **not** prove production accuracy, exact crash recovery, federated aggregation, or deployment compatibility. Those are separate concerns.

**Try next:** change the epoch count or learning rate in the YAML and rerun from the top. Compare the curve and validation score, not only whether the status says `completed`. The tutorial's accuracy assertions may intentionally fail for poor settings.